In [1]:
import pdfplumber

In [2]:
def extractedText(pdf_path):
    all_pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i,page in enumerate(pdf.pages):
            contentBox = (0,0,page.width,page.height*0.95)
            clean_page = page.within_bbox(contentBox)
            mylist = clean_page.extract_words(extra_attrs=['fontname','size'])
            all_pages.append(mylist)
    return all_pages

In [3]:
import re
def parseDoc(pages):
    pattern = r'^\d+\.\d+'
    for page in pages:
        for word in page:
            if "Bold" in word['fontname']:
                word["type"] = "heading"
            elif re.match(pattern, word['text']):
                word["type"] = "sub-heading"
            else:
                word["type"] = "body-text"
    return pages

In [12]:
from itertools import groupby
def group_into_lines(classified_pages):
    all_lines = []
    for page in classified_pages:
        sorted_words = sorted(page, key=lambda w: round(w['top'], 1))
        for top_val, group in groupby(sorted_words, key=lambda w: round(w['top'], 1)):
            line_words = list(group)
            # sort words left to right within the line
            line_words = sorted(line_words, key=lambda w: w['x0'])
            all_lines.append({
                "text": " ".join(w['text'] for w in line_words),
                "type": line_words[0]['type'],
                "top": top_val,
                "x0": line_words[0]['x0']
            })
    return all_lines

In [6]:
def group_into_sections(lines):
    sections = []
    current_section = None
    
    for line in lines:
        if line['type'] in ['heading', 'sub-heading']:
            if current_section:
                sections.append(current_section)
                # save the previous section
            # start a new section
            current_section = {
                'title': line['text'],
                'type': line['type'],
                'content': ''
            }
        else:
            if current_section:
                current_section['content']+=line['text']
                # add this line to current section content    
    # don't forget the last section
    if current_section:
        sections.append(current_section)
    
    return sections

In [7]:
import re
def extractMetaData(firstPage):
    text = firstPage['content']
    
    date_match = re.search(r'(\w+ \d{1,2},\s*\d{4})', text)
    date = datetime.strptime(date_match.group(1), "%B %d, %Y").strftime("%Y-%m") if date_match else "unknown"
    
    id_match = re.search(r'(CO\.\w+\.[\w./-]+)', text)
    doc_id = id_match.group(1) if id_match else Path(pdf_path).stem
    
    return doc_id, date

In [8]:
def clean_sections(sections):
    # find start index — first numbered section
    start = 0
    for i, section in enumerate(sections):
        if re.match(r'^\d+\.', section['title']):
            start = i
            break
    
    # find end index — first annexure
    end = len(sections)
    for i, section in enumerate(sections):
        if 'annex' in section['title'].lower() or 'table 1' in section['title'].lower():
            end = i
            break
    
    return sections[start:end]

In [9]:
import re
import chromadb
from datetime import datetime
from pathlib import Path
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings

# --- Step 1: Initialize embedding model ---
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# --- Step 2: Initialize ChromaDB collections ---
chroma_client = chromadb.PersistentClient(path="./vitta_setu_db")

rbi_parents_vectorstore = Chroma(
    client=chroma_client,
    collection_name="rbi_parents",
    embedding_function=embeddings
)

rbi_circulars_vectorstore = Chroma(
    client=chroma_client,
    collection_name="rbi_circulars",
    embedding_function=embeddings
)

# --- Step 3: Initialize document stores ---
parents_docstore = InMemoryStore()
circulars_docstore = InMemoryStore()

# --- Step 4: Initialize splitter ---
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

# --- Step 5: Initialize retrievers ---
parents_retriever = ParentDocumentRetriever(
    vectorstore=rbi_parents_vectorstore,
    docstore=parents_docstore,
    child_splitter=child_splitter,
)

circulars_retriever = ParentDocumentRetriever(
    vectorstore=rbi_circulars_vectorstore,
    docstore=circulars_docstore,
    child_splitter=child_splitter,
)

/Users/admin/Desktop/RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/qr/qd0wpk0524g91xk6t5nwg81h0000gn/T/ipykernel_26532/789145592.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
Loading weights: 100%|█████████████████████████████████████████████████████████████| 391/391 [00:00<00:00, 57814.74it/s]
/var/folders/qr/qd0wpk0524g91xk6t5nwg81h0000gn/T/ipykernel_26532/789145592.py:18: LangChainDeprecationWarning:

In [10]:
def ingest_document(pdf_path, doc_type, version="1.0"):
    # Step 1 — extract and reconstruct
    mylist = extractedText(pdf_path)
    classified = parseDoc(mylist)
    lines = group_into_lines(classified)
    sections = group_into_sections(lines)
    
    # Step 2 — extract metadata from cover page
    doc_id, date = extractMetaData(sections[0])
    
    # Step 3 — strip noise
    clean = clean_sections(sections)
    
    # Step 4 — convert sections to LangChain Documents
    documents = []
    for section in clean:
        full_text = section['title'] + "\n" + section['content']
        
        if not full_text.strip():
            continue
            
        doc = Document(
            page_content=full_text,
            metadata={
                "doc_id": doc_id,
                "doc_type": doc_type,
                "parent_id": doc_id,  # self reference for master directions
                "date": date,
                "section": section['title'],
                "version": version,
                "is_superseded": False,
                "conflict_detected": False
            }
        )
        documents.append(doc)
    
    # Step 5 — ingest into correct retriever
    if doc_type == "parent":
        parents_retriever.add_documents(documents)
    else:
        circulars_retriever.add_documents(documents)
    
    print(f"Ingested {len(documents)} sections from {doc_id}")
    return doc_id

In [13]:
ingest_document("/Users/admin/Downloads/ppi.PDF",'parent')

Ingested 163 sections from CO.DPSS.POLC.No.S-479/02.14.006/2021-22


'CO.DPSS.POLC.No.S-479/02.14.006/2021-22'

In [14]:
results = rbi_parents_vectorstore.similarity_search(
    "KYC requirements for full-KYC PPIs",
    k=3
)

for r in results:
    print(r.metadata['section'])
    print(r.page_content[:200])
    print("---")

9.2 Full-KYC PPIs
9.2 Full-KYC PPIs
---
8.2 PPIs for credit towards cross-border inward remittances
be loaded / reloaded in full-KYC PPIs issued to beneficiaries
---
2.8 Prepaid Payment Instruments (PPIs) : Instruments that facilitate purchase of goods and
. The features of theseinstruments are explained in paragraph 9.1 of this MD.(ii) Full-KYC PPIs : Issued by banks and non-banks after completing Know YourCustomer (KYC) of the PPI holder. These PPIs s
---


In [16]:
parent_results = parents_retriever.invoke(
    "KYC requirements for full-KYC PPIs"
)

for r in parent_results:
    print(r.metadata['section'])
    print(len(r.page_content), "chars")
    print(r.page_content[:300])
    print("---")

9.2 Full-KYC PPIs
3167 chars
9.2 Full-KYC PPIs
a) Banks and non-banks shall be permitted to issue such PPIs after completing KYC ofthe PPI holder (as indicated in paragraph 6);b) The Video-based Customer Identification Process (V-CIP), as detailed in Departmentof Regulation’s Master Direction on KYC dated February 25, 2016 (as 
---
8.2 PPIs for credit towards cross-border inward remittances
990 chars
8.2 PPIs for credit towards cross-border inward remittances
a) Bank and non-bank PPI issuer, appointed as Indian agent of authorised overseasprincipals, shall be permitted to issue full-KYC PPIs to beneficiaries of inwardremittances under the Money Transfer Services Scheme (MTSS) of RBI;b) Such PPIs
---
2.8 Prepaid Payment Instruments (PPIs) : Instruments that facilitate purchase of goods and
2438 chars
2.8 Prepaid Payment Instruments (PPIs) : Instruments that facilitate purchase of goods and
services, financial services, remittance facilities, etc., against the value stored therein. PPIst